# GL-FOPID Tuning via Reinforcement Learning (Gazebo)

**Runs on the laptop, against a real running Gazebo simulation** — this
notebook does not train anything by itself; it drives an already-running
`gazebo_sim.launch.py` session through many episodes.

**What this actually does**: searches for a single, static set of GL-FOPID
parameters (`Kp_theta`, `Ki_theta`, `Kd_theta`, `fopid_lambda`,
`fopid_mu`, `k_cross_track`) that minimizes row-tracking error and
IMU-detected chassis vibration when driving over floor thresholds, using
PPO (Stable-Baselines3) as the search algorithm. See
`docs/GL_FOPID_RL_TUNING_GUIDE.md` for the full reasoning, including an
honest note on why this is a heavier tool than the problem strictly
needs, and what to do if training doesn't converge in a reasonable time.

**Before running the real training cells**: run the smoke test (Step 0)
first. It costs a few seconds and confirms the whole pipeline — the
environment, the reward wiring, PPO itself — actually works, entirely
independent of Gazebo, before you commit real wall-clock time to a run
against the live simulation.

## Setup

In [ ]:
import sys, os, time
import numpy as np

# this notebook lives in src/strawberry_amr_gazebo/scripts/gl_fopid_rl/ --
# the modules it imports are in the same directory
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from reward import PARAM_NAMES, ACTION_DIM
from gazebo_env import GLFOPIDGazeboEnv, MockBackend

print("Tuning parameters:", PARAM_NAMES)
print("Action dimension:", ACTION_DIM)

## Step 0 — Smoke test (MockBackend, no Gazebo needed)

Confirms the environment, reward computation, and PPO training loop all
work together correctly, using a synthetic stand-in for Gazebo. This is
NOT a substitute for real training — `MockBackend` knows nothing about
your actual robot's dynamics — it exists purely to catch integration bugs
(a typo'd parameter name, a broken observation shape, an SB3 API mismatch)
in seconds rather than discovering them after 20 minutes of real,
wall-clock-expensive Gazebo episodes.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

smoke_env = GLFOPIDGazeboEnv(backend=MockBackend(seed=0), episode_duration_s=1.0)
check_env(smoke_env, warn=True)
print("check_env: PASSED")

smoke_model = PPO('MlpPolicy', smoke_env, verbose=0, n_steps=8, batch_size=8)
smoke_model.learn(total_timesteps=40)
print("Smoke-test PPO.learn() completed without error.")
print("Best (synthetic) reward found:", smoke_env._best_reward)
print("If this cell ran cleanly, the pipeline itself is sound -- any")
print("problems in the real training below are about the real system's")
print("behaviour, not a bug in this notebook's plumbing.")

## Step 1 — Generate the RL training world (with thresholds)

Run once. This world matches the shipped `row_navigation_params.yaml`
defaults (`--preset lab`) with two floor thresholds added — the "uneven
terrain / lab thresholds" the original training objective specifically
asks the controller to handle smoothly. Adjust `--thresholds` (comma-
separated x positions in metres from the row start) and
`--threshold-height` to match your own best estimate of Lab 3003's real
threshold geometry once you've measured it — these defaults are a
starting guess, not a verified measurement, same standard as every other
default in this project.

In [ ]:
import subprocess

result = subprocess.run([
    'python3', 'generate_polytunnel_world.py',
    '--preset', 'lab', '--num-rows', '1', '--row-length', '6.0',
    '--thresholds', '2.0,4.0', '--threshold-height', '0.012',
    '-o', '../worlds/irish_polytunnel_rl_training.world',
], cwd='..', capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("World generation failed -- see stderr above")

## Step 2 — Launch Gazebo (separate terminal, not this notebook)

Gazebo needs to run as its own long-lived process with a GUI window —
starting it from a notebook cell doesn't work well (the cell would block
for the entire simulation's lifetime, and Gazebo's window doesn't render
inside Jupyter). **Open a terminal and run this there, then come back
here**:

```bash
cd ~/strawberry_ws
ros2 launch strawberry_amr_gazebo gazebo_sim.launch.py \
    world:=$(ros2 pkg prefix strawberry_amr_gazebo)/share/strawberry_amr_gazebo/worlds/irish_polytunnel_rl_training.world \
    use_nav2:=false
```

`use_nav2:=false` — this training only needs `row_navigation` and the
sensor/actuation chain; Nav2 isn't part of what's being tuned and just
costs CPU you'd rather have free for training. Confirm Gazebo actually
came up (`ros2 topic echo /row_heading_error` should show numbers, not
silence) before running Step 3 below.

## Step 3 — Real training

**Time-boxing, stated honestly**: each episode takes `episode_duration_s`
seconds of real wall-clock time (Gazebo runs at or near real-time), plus
reset overhead. At the default 25s/episode, 500 episodes is roughly 3-4
hours of continuous real time — plan accordingly, and consider lowering
`total_episodes` for a first attempt rather than assuming a long run will
converge just because it runs longer.

**If reward isn't improving after a few dozen episodes**: that's a real,
useful signal, not something to push through by waiting longer. Check
`docs/GL_FOPID_RL_TUNING_GUIDE.md`'s troubleshooting section — the most
common causes are the abort thresholds being too tight (killing every
episode before it can show real performance differences) or the reward
weights not matching what you actually care about most.

In [ ]:
from gazebo_env import RealRosGazeboBackend
from stable_baselines3.common.callbacks import CheckpointCallback

TOTAL_EPISODES = 150   # start conservative -- see the time-boxing note above
CHECKPOINT_EVERY = 25

backend = RealRosGazeboBackend()   # connects to the ALREADY-RUNNING Gazebo from Step 2
env = GLFOPIDGazeboEnv(
    backend=backend,
    episode_duration_s=25.0,
    abort_lateral_m=0.35,
    abort_heading_rad=1.2,
    instability_penalty=5.0,
)

checkpoint_cb = CheckpointCallback(
    save_freq=CHECKPOINT_EVERY, save_path='./checkpoints/',
    name_prefix='gl_fopid_ppo')

model = PPO('MlpPolicy', env, verbose=1, n_steps=CHECKPOINT_EVERY,
            batch_size=CHECKPOINT_EVERY)
model.learn(total_timesteps=TOTAL_EPISODES, callback=checkpoint_cb)

print("Training complete.")
print("Best reward observed:", env._best_reward)
print("Best parameters observed:", env._best_params)

## Step 4 — Export the best parameters found

**Uses `env._best_params` — the single best PARAMETER SET actually
observed during training** (tracked continuously in `GLFOPIDGazeboEnv`,
see `gazebo_env.py`), not whatever PPO's final policy happens to output.
This matters: PPO's policy after `N` timesteps is not guaranteed to be
its best one — RL training can and does regress temporarily. Trusting the
best-observed result rather than the final one is a deliberate, safer
choice, not an oversight.

In [ ]:
import datetime

if env._best_params is None:
    raise RuntimeError("No completed episode yet -- run Step 3 first.")

out_path = f"gl_fopid_tuned_{datetime.date.today().isoformat()}.yaml"
with open(out_path, 'w') as f:
    f.write("# GL-FOPID parameters found by RL tuning (Stable-Baselines3 PPO)\n")
    f.write(f"# Generated: {datetime.datetime.now().isoformat()}\n")
    f.write(f"# Best reward observed: {env._best_reward:.4f}\n")
    f.write("# Copy these six values into src/row_navigation/config/row_navigation_params.yaml\n")
    f.write("# -- do NOT copy this whole file over the real config; it only has these six keys.\n")
    for name, value in env._best_params.items():
        f.write(f"{name}: {value:.4f}\n")

print(f"Wrote {out_path}")
print("")
print("NEXT STEP -- do not skip this: validate in Gazebo BEFORE the real")
print("robot. Manually copy these six values into")
print("src/row_navigation/config/row_navigation_params.yaml, rebuild, and")
print("re-run the bench test in ALL_IN_ONE_DEPLOYMENT_GUIDE.md Part 11 --")
print("in simulation first, then on the real hardware. An RL-found")
print("parameter set has not been validated against real chassis dynamics,")
print("real sensor noise, or real wheel/floor friction -- treat it as a")
print("strong starting point for the same bench-test process every other")
print("default value in this workspace goes through, not a finished result.")